In [0]:


DROP TABLE IF EXISTS silver.erp_cust_az12;

CREATE TABLE silver.erp_cust_az12
(
    cid INT,
    bdate DATE,
    gen STRING,
    ingestion_date TIMESTAMP
)
USING DELTA;


DROP TABLE IF EXISTS silver.erp_loc_a101;

CREATE TABLE silver.erp_loc_a101
(
    cid INT,
    cntry STRING,
    ingestion_date TIMESTAMP
)
USING DELTA;


DROP TABLE IF EXISTS silver.erp_px_cat_g1v2;

CREATE TABLE silver.erp_px_cat_g1v2
(
    id STRING,
    cat STRING,
    subcat STRING,
    maintenance STRING,
    ingestion_date TIMESTAMP
)
USING DELTA;



In [0]:
-- =========================================================================
-- Silver Layer: Bronze -> Silver 
-- Incremental and Cleansing (Deduplication, Normalization, Data Type Casting)
-- =========================================================================

In [0]:
-- STEP 1: SET RUNNING
UPDATE control.watermark
SET 
    last_run_status = 'RUNNING',
    last_run_start = current_timestamp()
WHERE table_name = 'erp_cust_az12'
AND layer = 'silver';

-- STEP 2: MERGE
MERGE INTO silver.erp_cust_az12 tgt
USING (

    WITH wm AS (
        SELECT last_load_timestamp
        FROM control.watermark
        WHERE table_name = 'erp_cust_az12'
        AND layer = 'silver'
    ),

    src AS (
        SELECT *
        FROM bronze.erp_cust_az12
        WHERE cid IS NOT NULL
        AND cid != 'CID'
        AND ingestion_date > (SELECT last_load_timestamp FROM wm)
    ),

    dedup AS (
        SELECT
            TRY_CAST(RIGHT(TRIM(cid), 5) AS INT) AS cid,
            CAST(bdate AS DATE) AS bdate,
            CASE 
                WHEN UPPER(TRIM(gen)) IN ('F','FEMALE') THEN 'Female'
                WHEN UPPER(TRIM(gen)) IN ('M','MALE') THEN 'Male'
                ELSE 'n/a'
            END AS gen,
            ingestion_date,
            ROW_NUMBER() OVER (PARTITION BY cid ORDER BY ingestion_date DESC) rn
        FROM src
    )

    SELECT
        cid, bdate, gen, ingestion_date
    FROM dedup WHERE rn = 1

) src
ON tgt.cid = src.cid

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- STEP 3: UPDATE WATERMARK (END TIME)
UPDATE control.watermark
SET 
    last_load_timestamp = (SELECT MAX(ingestion_date) FROM bronze.erp_cust_az12),
    last_run_status = 'SUCCESS',
    last_run_end = current_timestamp(),
    updated_at = current_timestamp()
WHERE table_name = 'erp_cust_az12'
AND layer = 'silver';

In [0]:
-- STEP 1
UPDATE control.watermark
SET last_run_status='RUNNING',
    last_run_start=current_timestamp()
WHERE table_name='erp_loc_a101' AND layer='silver';

-- STEP 2
MERGE INTO silver.erp_loc_a101 tgt
USING (

    WITH wm AS (
        SELECT last_load_timestamp
        FROM control.watermark
        WHERE table_name='erp_loc_a101' AND layer='silver'
    ),

    src AS (
        SELECT *
        FROM bronze.erp_loc_a101
        WHERE cid IS NOT NULL
        AND cid != 'CID'
        AND ingestion_date > (SELECT last_load_timestamp FROM wm)
    ),

    dedup AS (
        SELECT
            TRY_CAST(RIGHT(TRIM(cid), 5) AS INT) AS cid,
            TRIM(cntry) cntry,
            ingestion_date,
            ROW_NUMBER() OVER (PARTITION BY cid ORDER BY ingestion_date DESC) rn
        FROM src
    )

    SELECT
        cid, cntry, ingestion_date
    FROM dedup WHERE rn=1

) src
ON tgt.cid=src.cid

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- STEP 3
UPDATE control.watermark
SET last_load_timestamp=(SELECT MAX(ingestion_date) FROM bronze.erp_loc_a101),
last_run_status='SUCCESS',
last_run_end=current_timestamp(),
updated_at=current_timestamp()
WHERE table_name='erp_loc_a101' AND layer='silver';

In [0]:
-- STEP 1
UPDATE control.watermark
SET last_run_status='RUNNING',
    last_run_start=current_timestamp()
WHERE table_name='erp_px_cat_g1v2' AND layer='silver';

-- STEP 2
MERGE INTO silver.erp_px_cat_g1v2 tgt
USING (

    WITH wm AS (
        SELECT last_load_timestamp
        FROM control.watermark
        WHERE table_name='erp_px_cat_g1v2' AND layer='silver'
    ),

    src AS (
        SELECT *
        FROM bronze.erp_px_cat_g1v2
        WHERE ingestion_date > (SELECT max(last_load_timestamp) FROM wm)
    ),

    dedup AS (
        SELECT
            NULLIF(TRIM(id), '')  AS id,
            TRIM(cat) cat,
            TRIM(subcat) subcat,
            TRIM(maintenance) maintenance,
            ingestion_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY ingestion_date DESC) rn
        FROM src
    )

    SELECT
        id, cat, subcat, maintenance, ingestion_date
    FROM dedup WHERE rn=1

) src
ON tgt.id=src.id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- STEP 3
UPDATE control.watermark
SET last_load_timestamp=(SELECT MAX(ingestion_date) FROM bronze.erp_px_cat_g1v2),
last_run_status='SUCCESS',
last_run_end=current_timestamp(),
updated_at=current_timestamp()
WHERE table_name='erp_px_cat_g1v2' AND layer='silver';